In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

/home/aromal/anaconda3/envs/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [6]:
from dotenv import load_dotenv

In [7]:
model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

In [ ]:
vector_store = Chroma(
    embedding_function=model,
    persist_directory="chroma_db",
    collection_name="sample"
)

### add documents

In [9]:
vector_store.add_documents(docs)

['7fe4f941-3751-4b52-bf22-3b88456eaf95',
 'e758ac76-69e9-4789-a163-93f30ef4dbc3',
 '0e234034-e414-4a2f-9c24-29d5a14ffeab',
 '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
 '132cdcd8-2af4-456a-9125-7c990f384869']

In [ ]:
vector_store.get(include=['embeddings', 'documents', 'metadatas']) # should be metadatas not metadata

{'ids': ['7fe4f941-3751-4b52-bf22-3b88456eaf95',
  'e758ac76-69e9-4789-a163-93f30ef4dbc3',
  '0e234034-e414-4a2f-9c24-29d5a14ffeab',
  '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
  '132cdcd8-2af4-456a-9125-7c990f384869'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

### search documents

In [12]:
# search document
vector_store.similarity_search(
    query="Who among these are a bowler",
    k=2
)

[Document(id='03ea6536-5020-4fa2-b4e8-ed81a23c1039', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='132cdcd8-2af4-456a-9125-7c990f384869', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [13]:
# search document with score
vector_store.similarity_search_with_score(
    query="Who among these are a bowler",
    k=2
)

[(Document(id='03ea6536-5020-4fa2-b4e8-ed81a23c1039', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6391493082046509),
 (Document(id='132cdcd8-2af4-456a-9125-7c990f384869', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6609402894973755)]

In [14]:

# We can also filer on metadata
vector_store.similarity_search_with_score(
    query="Who among these are a bowler",
    k=2,
    filter={"team":"Chennai Super Kings"}
)

[(Document(id='132cdcd8-2af4-456a-9125-7c990f384869', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6609402894973755),
 (Document(id='0e234034-e414-4a2f-9c24-29d5a14ffeab', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.725570559501648)]